# Run Spectral Ratio Illumination Demo on Google Colab

This notebook allows you to run the project on Google Colab with GPU support.

## Setup Steps:
1. **Enable GPU Runtime**: Runtime → Change runtime type → Hardware accelerator → **GPU** → Save
2. **Run all cells in order** (Runtime → Run all)
3. **Upload model to Google Drive** at the path shown below (or provide a download link)

## What this notebook does:
- Checks GPU availability
- Mounts Google Drive for model storage
- Clones your GitHub repository
- Installs all dependencies (PyTorch with GPU/CPU support)
- Runs validation checks
- Processes images with your algorithms
- Downloads results as a .tar.gz file

## Step 1: Check GPU Availability

In [ ]:
# Check if GPU is available
!nvidia-smi || echo "⚠️ No GPU detected - will use CPU (slower but works)"

## Step 2: Mount Google Drive

**Important**: After running this cell, you'll need to upload the model file to:

`/MyDrive/Spectral_Ratio_Illumination_Demo/model/UNET_run_x10_01_last_model.pth`

You can:
- Upload the 528MB model file to this Drive location
- Or ask your professor for a shared Drive link to the model
- Or skip the model and run baseline Retinex only (see instructions later)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully")

## Step 3: Clone GitHub Repository

In [ ]:
import os

# Remove if already exists from previous runs
if os.path.exists('/content/Spectral_Ratio_Illumination_Demo'):
    !rm -rf /content/Spectral_Ratio_Illumination_Demo

# Clone the repository
!git clone https://github.com/oelmady/Spectral_Ratio_Illumination_Demo.git /content/Spectral_Ratio_Illumination_Demo

# Change to the project directory
%cd /content/Spectral_Ratio_Illumination_Demo

# Show directory structure
!ls -la
print("\n✓ Repository cloned successfully")

## Step 4: Copy Model from Google Drive

This cell copies the model checkpoint from your Google Drive to the project folder.

**If you don't have the model yet:**
- You can skip this and run baseline Retinex only (no neural network)
- Ask your professor for a shared link to the model file
- Upload it manually to Drive at the path shown below

In [ ]:
import os
from shutil import copyfile

# Path to model in Google Drive
drive_model = '/content/drive/MyDrive/Spectral_Ratio_Illumination_Demo/model/UNET_run_x10_01_last_model.pth'

# Path to model in project folder
local_model_dir = '/content/Spectral_Ratio_Illumination_Demo/model'
local_model = os.path.join(local_model_dir, 'UNET_run_x10_01_last_model.pth')

# Create model directory if it doesn't exist
os.makedirs(local_model_dir, exist_ok=True)

# Copy model from Drive if it exists
if os.path.exists(drive_model):
    print(f"📥 Copying model from Drive...")
    copyfile(drive_model, local_model)
    size_mb = os.path.getsize(local_model) / (1024 * 1024)
    print(f"✓ Model copied successfully ({size_mb:.1f} MB)")
    print(f"   Location: {local_model}")
else:
    print("⚠️ Model not found in Google Drive at:")
    print(f"   {drive_model}")
    print("\n📝 You can either:")
    print("   1. Upload the model to that Drive location and re-run this cell")
    print("   2. Continue without the model (baseline Retinex only)")

## Step 5: Install Dependencies

This cell installs all required Python packages:
- PyTorch (GPU version if available, otherwise CPU)
- OpenCV (headless version for no GUI issues)
- NumPy, Matplotlib, and other dependencies

In [ ]:
import subprocess
import sys

print("📦 Installing dependencies...\n")

# Upgrade pip
print("1️⃣ Upgrading pip...")
!pip install --quiet --upgrade pip setuptools wheel

# Install opencv-headless and basic dependencies
print("\n2️⃣ Installing OpenCV, NumPy, Matplotlib...")
!pip install --quiet opencv-python-headless numpy matplotlib

# Install PyTorch with GPU support if available
print("\n3️⃣ Installing PyTorch...")
try:
    subprocess.run(["nvidia-smi"], check=True, capture_output=True)
    print("   GPU detected: Installing CUDA-enabled PyTorch (cu118)")
    !pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cu118
except:
    print("   No GPU: Installing CPU-only PyTorch")
    !pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cpu

# Install any remaining requirements
print("\n4️⃣ Installing remaining requirements...")
!pip install --quiet -r requirements.txt 2>/dev/null || true

print("\n✓ All dependencies installed successfully!")

# Verify installations
print("\n📋 Checking installed versions:")
import torch
import cv2
import numpy as np
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")
print(f"   OpenCV: {cv2.__version__}")
print(f"   NumPy: {np.__version__}")

## Step 6: Run Preflight Check

This validates that everything is set up correctly.

In [ ]:
%cd /content/Spectral_Ratio_Illumination_Demo
!python preflight_check.py

## Step 7: Run Experiments

This cell processes all images in `data/images/` with your algorithms.

**Two modes:**
1. **With model** (if you uploaded it): Runs neural ISD prediction + all Retinex variants
2. **Without model**: Runs baseline and SR-constrained Retinex only

The cell automatically detects which mode to use.

In [ ]:
import os

%cd /content/Spectral_Ratio_Illumination_Demo

model_path = '/content/Spectral_Ratio_Illumination_Demo/model/UNET_run_x10_01_last_model.pth'

if os.path.exists(model_path) and os.path.getsize(model_path) > 1000000:  # > 1MB
    print("🚀 Running FULL experiment (with neural ISD model)...")
    print("   This includes: model inference + SR-Retinex + baseline Retinex + color correction\n")
    !python scripts/run_batch.py \
        --use-model \
        --retinex \
        --baseline-retinex \
        --sr-correct \
        --iterations 5 \
        --sigma 15 \
        --distance 1.0
else:
    print("🚀 Running BASELINE experiment (no model)...")
    print("   This includes: baseline Retinex + SR-constrained Retinex (using annotated maps)\n")
    print("   ⚠️ Note: Since no model is available, this will use pre-existing SR maps")
    print("   from data/sr_maps/ if available, or skip SR-constrained processing.\n")
    !python scripts/run_batch.py \
        --baseline-retinex \
        --retinex \
        --iterations 5 \
        --sigma 15

print("\n✓ Processing complete! Check results/ directory.")

## Step 8: View Sample Results (Optional)

Display a few output images to verify processing worked correctly.

In [ ]:
import os
import glob
from IPython.display import Image, display
import matplotlib.pyplot as plt

# Find PNG outputs in results directory
png_files = glob.glob('/content/Spectral_Ratio_Illumination_Demo/results/*.png')

if png_files:
    print(f"📸 Found {len(png_files)} output images. Showing first 3:\n")
    for img_path in png_files[:3]:
        print(f"   {os.path.basename(img_path)}")
        display(Image(filename=img_path, width=600))
        print()
else:
    print("⚠️ No PNG outputs found in results/")
    print("   Check if processing completed successfully above.")

## Step 9: Package and Download Results

This creates a `.tar.gz` archive of all results and downloads it to your local machine.

In [ ]:
import os
from google.colab import files

%cd /content/Spectral_Ratio_Illumination_Demo

# Create archive of results
print("📦 Packaging results...")
!tar -czf results.tar.gz results/ 2>/dev/null || echo "No results directory found"

if os.path.exists('results.tar.gz'):
    size_mb = os.path.getsize('results.tar.gz') / (1024 * 1024)
    print(f"✓ Archive created: results.tar.gz ({size_mb:.1f} MB)")
    print("\n⬇️ Downloading to your computer...")
    files.download('results.tar.gz')
    print("✓ Download complete!")
    print("\nTo extract on your Mac:")
    print("   tar -xzf results.tar.gz")
else:
    print("⚠️ No results to package. Make sure Step 7 completed successfully.")

---

## Optional: Advanced Parameter Tuning

Run multiple experiments with different parameters to find optimal settings.

**Note**: This cell is optional. Only run it if you want to experiment with different parameter values.

In [ ]:
# Example: Test different sigma values (blur strength)
# Uncomment and run to execute parameter sweep

# %cd /content/Spectral_Ratio_Illumination_Demo
# 
# for sigma in [10, 15, 20, 25, 30]:
#     print(f"\n{'='*60}")
#     print(f"Testing with sigma={sigma}")
#     print('='*60)
#     !python scripts/run_batch.py \
#         --baseline-retinex \
#         --retinex \
#         --iterations 5 \
#         --sigma {sigma}
# 
# print("\n✓ Parameter sweep complete!")

# You can also test different iteration counts or distance values
# See TUNING_GUIDE.md in the repo for recommended parameter ranges

---

## Troubleshooting

### Model not found error
- Upload the model to Google Drive at: `/MyDrive/Spectral_Ratio_Illumination_Demo/model/UNET_run_x10_01_last_model.pth`
- Re-run Step 4 to copy it
- Or continue without model (baseline Retinex only)

### Out of memory error
- Switch to CPU runtime (Runtime → Change runtime type → None)
- Or process fewer images at once

### Import errors
- Re-run Step 5 (dependency installation)
- Make sure PyTorch and OpenCV installed successfully

### No results generated
- Check that `data/images/` contains .tif or .tiff files
- Verify Step 7 ran without errors
- Check error messages in the output

### Need help?
- See `TUNING_GUIDE.md` and `README_EXPERIMENTS.md` in the repository
- Check `preflight_check.py` output for specific issues